In [1]:
import pygame
import numpy as np
import time
import sys

from tetris_env import TetrisEnv
from keras_dqn_agent import DQNAgent
from tetris_keras import Tetris

pygame 2.1.3 (SDL 2.0.22, Python 3.11.14)
Hello from the pygame community. https://www.pygame.org/contribute.html


2026-04-21 15:25:21.104979: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-04-21 15:25:21.158751: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-21 15:25:22.630175: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-21 15:25:26.649718: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To tur

In [2]:
# Config 
ROWS         = 14
COLS         = 8
WEIGHTS_PATH = "tetris_dqn.weights.h5"
HIDDEN_UNITS = [128, 64, 32]

# Display
CELL        = 40          # px per cell
MARGIN      = 1           # px gap between cells
SIDEBAR_W   = 220         # info panel width
TOP_PAD     = 20
BOT_PAD     = 20
LEFT_PAD    = 20

STEP_DELAY  = 0.35        # seconds between placements (lower = faster)
NUM_EPISODES = 5

# Colors
C_BG        = (18,  18,  30)
C_GRID      = (40,  40,  60)
C_EMPTY     = (28,  28,  45)
C_SIDEBAR   = (24,  24,  40)
C_TEXT      = (220, 220, 240)
C_DIM       = (120, 120, 150)
C_ACCENT    = (100, 180, 255)
C_BORDER    = (60,  60,  90)

# Piece colors — one per piece ID (I T L J Z S O)
PIECE_COLORS = [
    (0,   240, 240),   # I — cyan
    (160,  0,  240),   # T — purple
    (240, 160,   0),   # L — orange
    (0,   80,  240),   # J — blue
    (240,  40,  40),   # Z — red
    (0,   240,  80),   # S — green
    (240, 240,   0),   # O — yellow
]
C_GHOST     = (80,  80, 110)   # ghost / locked piece tint
C_ACTIVE    = None             # use PIECE_COLORS[id]

In [ ]:
class TetrisRenderer:
    """
    Draws the Tetris board, the active piece, the next piece preview,
    and a stats sidebar using PyGame.
    """

    def __init__(self, rows: int, cols: int):
        self.rows = rows
        self.cols = cols

        board_w = cols * (CELL + MARGIN) + MARGIN
        board_h = rows * (CELL + MARGIN) + MARGIN
        win_w   = LEFT_PAD + board_w + SIDEBAR_W + LEFT_PAD
        win_h   = TOP_PAD  + board_h + BOT_PAD

        pygame.init()
        pygame.display.set_caption("Tetris DQN Agent")
        self.screen = pygame.display.set_mode((win_w, win_h))
        self.font_lg = pygame.font.SysFont("monospace", 22, bold=True)
        self.font_md = pygame.font.SysFont("monospace", 17)
        self.font_sm = pygame.font.SysFont("monospace", 14)
        self.clock  = pygame.time.Clock()

        # board origin (top-left of cell grid)
        self.ox = LEFT_PAD
        self.oy = TOP_PAD
        self.board_w = board_w
        self.board_h = board_h

    def _cell_rect(self, col: int, row: int) -> pygame.Rect:
        x = self.ox + MARGIN + col * (CELL + MARGIN)
        y = self.oy + MARGIN + row * (CELL + MARGIN)
        return pygame.Rect(x, y, CELL, CELL)

    def _draw_cell(self, col: int, row: int, color, highlight: bool = False):
        rect = self._cell_rect(col, row)
        pygame.draw.rect(self.screen, color, rect, border_radius=4)
        if highlight:
            bright = tuple(min(255, c + 60) for c in color)
            pygame.draw.rect(self.screen, bright, rect, width=2, border_radius=4)

    def _draw_mini_cell(self, surf, col: int, row: int, color, cell_size: int = 22):
        gap = 2
        x = gap + col * (cell_size + gap)
        y = gap + row * (cell_size + gap)
        r = pygame.Rect(x, y, cell_size, cell_size)
        pygame.draw.rect(surf, color, r, border_radius=3)

    def _draw_board(self, board_copy: np.ndarray, piece_id: int):
        # Background
        bg = pygame.Rect(self.ox, self.oy, self.board_w, self.board_h)
        pygame.draw.rect(self.screen, C_BORDER, bg, border_radius=4)

        piece_color  = PIECE_COLORS[piece_id]

        for r in range(self.rows):
            for c in range(self.cols):
                v = board_copy[r, c]
                if v == 0:
                    self._draw_cell(c, r, C_EMPTY)
                elif v == 1:
                    self._draw_cell(c, r, C_GHOST, highlight=False)
                elif v == 2:  # active piece
                    self._draw_cell(c, r, piece_color, highlight=True)

    def _draw_next(self, sx: int, sy: int, next_id: int):
        label = self.font_sm.render("NEXT", True, C_DIM)
        self.screen.blit(label, (sx, sy))
        sy += 22

        cell_size = 22
        gap       = 2
        box_w     = 4 * (cell_size + gap) + gap
        box_h     = 4 * (cell_size + gap) + gap
        surf      = pygame.Surface((box_w, box_h), pygame.SRCALPHA)
        surf.fill((35, 35, 55))

        coords = Tetris.tetris_pieces[next_id][0]
        color  = PIECE_COLORS[next_id]
        for x, y in coords:
            self._draw_mini_cell(surf, x, y, color, cell_size)

        self.screen.blit(surf, (sx, sy))
        return sy + box_h + 12

    def _draw_sidebar(self, stats: dict, next_id: int):
        sx = self.ox + self.board_w + 18
        sy = self.oy

        # Panel
        panel = pygame.Rect(sx - 8, sy, SIDEBAR_W - 10, self.board_h)
        pygame.draw.rect(self.screen, C_SIDEBAR, panel, border_radius=6)

        sx += 4
        sy += 12

        # Title
        t = self.font_lg.render("DQN AGENT", True, C_ACCENT)
        self.screen.blit(t, (sx, sy))
        sy += 34

        # Next piece
        sy = self._draw_next(sx, sy, next_id)
        sy += 6

        # Divider
        pygame.draw.line(self.screen, C_BORDER, (sx, sy), (sx + SIDEBAR_W - 30, sy), 1)
        sy += 14

        # Stats
        rows = [
            ("Episode",  f"{stats.get('episode', 1)} / {stats.get('total_eps', NUM_EPISODES)}"),
            ("Pieces",   str(stats.get('pieces', 0))),
            ("Lines",    str(stats.get('lines',  0))),
            ("Reward",   f"{stats.get('reward', 0.0):.1f}"),
        ]
        for label, value in rows:
            lbl = self.font_sm.render(label, True, C_DIM)
            val = self.font_md.render(value, True, C_TEXT)
            self.screen.blit(lbl, (sx, sy))
            self.screen.blit(val, (sx, sy + 16))
            sy += 46

    # Main render 

    def render(self, board_copy: np.ndarray, piece_id: int, next_id: int, stats: dict):
        self.screen.fill(C_BG)
        self._draw_board(board_copy, piece_id)
        self._draw_sidebar(stats, next_id)
        pygame.display.flip()
        self.clock.tick(60)

    def close(self):
        pygame.quit()

In [4]:
# Demo Loop
def run_demo(
    episodes: int    = NUM_EPISODES,
    step_delay: float = STEP_DELAY,
    weights_path: str = WEIGHTS_PATH,
):
    """
    Opens a PyGame window and plays games using the trained DQN agent.
    """
    env      = TetrisEnv(rows=ROWS, cols=COLS, render=False)
    agent    = DQNAgent(feature_size=env.feature_size, hidden_units=HIDDEN_UNITS)
    renderer = TetrisRenderer(ROWS, COLS)

    try:
        agent.load(weights_path)
        agent.epsilon = 0.0   # pure greedy for demo
    except Exception as e:
        print(f"[demo] Could not load weights from '{weights_path}': {e}")
        print("       Running with random weights instead.")

    def _handle_events() -> bool:
        """Returns False if the user wants to quit."""
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                return False
            if event.type == pygame.KEYDOWN:
                if event.key in (pygame.K_ESCAPE, pygame.K_q):
                    return False
        return True

    running = True

    for ep in range(1, episodes + 1):
        ts           = env._reset()
        total_reward = 0.0

        while True:
            # PyGame event pump
            if not _handle_events():
                running = False
                break

            # Get valid placements
            next_states = env.get_next_states()
            if not next_states:
                break

            action = agent.act_greedy(next_states)
            ts     = env._step(action)
            total_reward += float(ts.reward)

            # Build stats dict for the sidebar
            stats = {
                "pieces":    env._game.pieces_placed,
                "lines":     env._game.score,
                "reward":    total_reward
            }

            renderer.render(
                board_copy = env._game.get_board_copy(),
                piece_id   = env._game.current_piece,
                next_id    = env._game.next_piece,
                stats      = stats,
            )

            time.sleep(step_delay)

            if ts.is_last():
                break

        if not running:
            break

        # Brief pause between episodes
        if running:
            pause_start = time.time()
            while time.time() - pause_start < 1.0:
                if not _handle_events():
                    running = False
                    break
                renderer.clock.tick(30)

    renderer.close()

In [5]:
run_demo(
    episodes   = 5,
    step_delay = 0.35,
    weights_path = "tetris_dqn.weights.h5",
)

2026-04-21 15:25:30.488546: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


[Agent] Loaded: tetris_dqn.weights.h5
Pieces Placed: 121
Pieces Placed: 52
